In [1]:
from IPython.display import Image

### prefill vs decode

- prefill
    - 矩阵与矩阵相乘（matrix-matrix multiplication），这种运算模式能够高效地利用现代GPU中的张量核心（Tensor Cores），使其达到较高的计算吞吐量。因此，预填充阶段通常是计算密集型（compute-bound）的，其速度相对较快 。
    - 在投机采样中，target model 的 verification 阶段，将这个包含n个候选词元的序列作为一个整体，输入到原始的、更大更强的“目标模型”（Target Model）中。目标模型对这n个词元进行一次并行的前向传播。这个验证步骤的计算模式类似于高效的Prefill阶段，将多个低效的矩阵-向量运算合并成一个高效的矩阵-矩阵运算，从而有效利用GPU的并行计算能力 。
- decode
    - 串行自回归
    - 与预填充阶段不同，解码阶段的每次计算都是一个矩阵与向量相乘（matrix-vector operation），这种操作无法充分利用GPU的并行计算能力，导致其性能受限于内存带宽而非计算能力 。这一阶段通常是内存带宽密集型（Memory-Bandwidth Bound），其速度更多地受限于从 GPU 显存中读写数据的速率，而非计算单元的性能。 

### kv cache => Prompt Caching

Prompt caching 之 前缀缓存，将KV缓存概念扩展到多个请求间的技术。通过缓存共享前缀（如系统提示词）的KV缓存，新请求可以跳过重复计算，直接复用缓存结果，显著减少计算负载并加速推理。

In [2]:
Image(url='./imgs/prompt-caching.jpeg', width=400)

$$
\text{Total KV Cache Size (bytes)}=\text{batch\_size} \times (\text{sequence\_length}) \times 2 \times (\text{num\_layers}) \times (\text{hidden\_size}) \times \text{sizeof(precision)}
$$
- KV Caching 的核心洞察在于：在自回归解码过程中，对于任何已经生成的令牌，其 K 向量和 V 向量是固定不变的 。当模型生成新令牌时，无需重新计算整个历史序列的 K 和 V 向量。取而代之，系统可以将这些计算出的 K 和 V 张量存储在高速的 GPU 显存中。在下一步解码时，模型只需为最新的一个令牌计算其 Q、K、V 向量，然后用新的 Q 向量与缓存中所有历史 K 向量计算注意力，再加权缓存中的 V 向量。
- Prompt Caching，也被称为前缀缓存（Prefix Caching）或上下文缓存（Context Caching），是 KV Caching 技术在更高层次上的应用。如果说 KV Caching 优化的是单次生成请求内部的计算，那么 Prompt Caching 优化的则是多次独立请求之间的计算 。
    - 当一个 API 请求包含一个较长的、可能被重复使用的前缀（例如，一段详细的系统指令、一篇用于问答的文档）时，推理系统在处理这个请求后，会将该前缀对应的 KV Cache 状态（即所有前缀令牌的 K 和 V 张量）保存下来。当后续有新的请求，其提示的起始部分与已保存的前缀完全一致时，系统可以直接从缓存中加载这个预先计算好的 KV Cache 状态，从而完全跳过对这部分共享前缀的预填充计算，直接从前缀末尾开始解码。
- vLLM 实现了一个名为“自动前缀缓存”（Automatic Prefix Caching, APC）的功能。其设计目标是成为一项“免费的午餐”式优化，即在不改变任何模型输出的前提下，自动提升服务性能 。
    - 激活方式：通过在初始化 LLM 引擎时设置 enable_prefix_caching=True 标志即可启用此功能。
    - 典型用例：vLLM 的文档明确指出，APC 在处理长文档问答和多轮对话等场景中能发挥巨大作用。在这些场景下，不断增长的对话历史构成了一个共享的前缀，可以被高效复用。
- Prompt Caching 的引入，使得提示工程（Prompt Engineering）不再仅仅是关于模型输出质量的艺术，更成为一门关乎系统性能和成本优化的科学。为了最大化缓存命中率，开发者应遵循以下核心原则：
    - 黄金法则：静态前缀，动态后缀：精心设计提示结构，将不变的、可复用的内容（如系统指令、背景信息、少量示例）放在提示的开头，形成一个稳定的长前缀；而将每次请求都变化的、用户特定的内容（如具体问题、变量）放在提示的末尾，形成一个简短的动态后缀 。
    - 具体实践示例：
        - 系统指令与角色扮演：将详细的系统指令、角色定义、输出格式要求等置于最前端 。
        - 少量示例学习 (Few-shot Learning)：将用于指导模型的示例（Examples）作为静态前缀的一部分 。
        - 工具与函数定义：在构建智能体或使用函数调用时，将工具（Tools）的定义和描述放在前缀中进行缓存 。
        - 多模态输入：对于包含图像的提示，确保作为前缀的图像部分在多次请求中保持完全一致 。

### cache 管理

- https://manus.im/zh-cn/blog/Context-Engineering-for-AI-Agents-Lessons-from-Building-Manus
    - https://platform.openai.com/docs/guides/prompt-caching
    - https://openai.com/index/api-prompt-caching/
        - session: "Caches are typically cleared after **5-10 minutes of inactivity** and are always removed within one hour of the cache's last use. As with all API services, Prompt Caching is subject to our Enterprise privacy⁠ commitments. Prompt caches are not shared between organizations."